## Análise Explorotária de *olist_orders_dataset.csv*

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("./olist_orders_dataset.csv")

In [65]:
df.sample(5)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
74362,851b8de268e532c930dc8f315d1434d0,ea8b0305292d6ec9236d154b122707ea,delivered,2017-10-20 16:31:11,2017-10-20 16:50:11,2017-10-23 20:21:22,2017-10-27 18:33:20,2017-11-10
52002,17b3a4d85ba0793bf46aa81893c3f95b,c27808faa6aa8b89eba6c04408025c68,delivered,2018-01-14 23:59:15,2018-01-15 17:31:37,2018-01-16 20:43:01,2018-01-17 15:56:41,2018-01-30
54431,008720830a8a30e39e6f22f81818a141,f1af79ed0c452b79fec6a0217f60a6da,delivered,2017-08-13 16:44:57,2017-08-13 16:55:17,2017-08-14 16:12:41,2017-08-18 10:24:01,2017-08-25
66075,36a179cea5792575e6b9682f13e44287,5e74f2276d4494b40c013f43a6584339,delivered,2017-07-16 10:35:24,2017-07-18 06:15:35,2017-07-20 16:59:57,2017-09-19 15:54:40,2017-08-16
45747,9f1d14a57c14b6fff699b45a39d60bc5,39302f15abb7e40328b03be978884590,delivered,2017-03-21 19:03:48,2017-03-26 23:32:42,2017-03-30 16:25:25,2017-04-10 14:49:44,2017-04-20


### Conversão de tipos

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [70]:
colunas_de_data = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

df[colunas_de_data] = df[colunas_de_data].apply(pd.to_datetime , errors = "coerce")

df[colunas_de_data].dtypes

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

### Análise de nulos

In [71]:
df.isnull().mean().mul(100).round(2).sort_values(ascending = False)

order_delivered_customer_date    2.98
order_delivered_carrier_date     1.79
order_approved_at                0.16
order_id                         0.00
order_purchase_timestamp         0.00
order_status                     0.00
customer_id                      0.00
order_estimated_delivery_date    0.00
dtype: float64

In [25]:
df.groupby("order_status")[colunas_de_data].apply(lambda x: x.isnull().mean()).mul(100).round(2)

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
order_status,,,,,
approved,0.0,0.00,100.0,100.00,0.0
canceled,0.0,22.56,88.0,99.04,0.0
created,0.0,100.00,100.0,100.00,0.0
delivered,0.0,0.01,0.0,0.01,0.0
invoiced,0.0,0.00,100.0,100.00,0.0
processing,0.0,0.00,100.0,100.00,0.0
shipped,0.0,0.00,0.0,100.00,0.0
unavailable,0.0,0.00,100.0,100.00,0.0


### Análise de possíveis chaves e cardinalidade

In [40]:
len(df)

99441

In [41]:
df.nunique().sort_values()

order_status                         8
order_estimated_delivery_date      459
order_delivered_carrier_date     81018
order_approved_at                90733
order_delivered_customer_date    95664
order_purchase_timestamp         98875
order_id                         99441
customer_id                      99441
dtype: int64

In [48]:
'''
1.0       -> Identificador (PK)
0.8 - 1.0 -> Relacionamento
< 0.8     -> Categoria
'''
(df.nunique().sort_values() / len(df)).sort_values(ascending = False)

order_id                         1.000000
customer_id                      1.000000
order_purchase_timestamp         0.994308
order_delivered_customer_date    0.962018
order_approved_at                0.912430
order_delivered_carrier_date     0.814734
order_estimated_delivery_date    0.004616
order_status                     0.000080
dtype: float64

### Análise consistência temporal:
Se valor retornado for maior que zero .: problemas estruturais nos dados nos dados

In [52]:
(df["order_purchase_timestamp"] > df["order_approved_at"]).sum()

np.int64(0)

In [53]:
(df["order_approved_at"] > df["order_delivered_carrier_date"]).sum()

np.int64(1359)

In [57]:
mask = df["order_approved_at"] > df["order_delivered_carrier_date"]
df[mask].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
15,dcb36b511fcac050b97cd5c05de84dc3,3b6828a50ffe546942b7a473d70ac0fc,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32,2018-07-04
64,688052146432ef8253587b930b01a06d,81e08b08e5ed4472008030d70327c71f,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14,2018-04-24 19:31:58,2018-05-15
199,58d4c4747ee059eeeb865b349b41f53a,1755fad7863475346bc6c3773fe055d3,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00,2018-07-25 23:58:19,2018-07-31
210,412fccb2b44a99b36714bca3fef8ad7b,c6865c523687cb3f235aa599afef1710,delivered,2018-07-22 22:30:05,2018-07-23 12:31:53,2018-07-23 12:24:00,2018-07-24 19:26:42,2018-07-31
415,56a4ac10a4a8f2ba7693523bb439eede,78438ba6ace7d2cb023dbbc81b083562,delivered,2018-07-22 13:04:47,2018-07-27 23:31:09,2018-07-24 14:03:00,2018-07-28 00:05:39,2018-08-06


In [58]:
(df["order_delivered_carrier_date"] > df["order_delivered_customer_date"]).sum()

np.int64(23)

In [60]:
mask = df["order_delivered_carrier_date"] > df["order_delivered_customer_date"]
df[mask].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,5f50465da00b7fed5dd1239f4ecf6e2c,delivered,2017-07-20 11:20:52,2017-07-21 06:43:14,2017-07-28 16:57:58,2017-07-25 19:32:56,2017-08-14
9553,383aa8b2724fe452d9ccd9934a8c628b,b1cb2f9d7a19480f3749e248db14d58f,delivered,2017-07-02 20:58:43,2017-07-02 21:10:20,2017-07-07 17:22:41,2017-07-06 14:27:51,2017-07-21
13487,cb1134f9010d242e9515ad1c78ec0c39,2fd33ac77677bd214b1882868317eeed,delivered,2017-07-16 12:35:34,2017-07-18 06:03:50,2017-07-20 19:22:02,2017-07-19 14:13:28,2017-08-08
14474,dceb62e8fa94b46006c9554fed743df0,2721900eb4e0f1cc2c836dd7bc1b1e11,delivered,2017-07-20 20:58:05,2017-07-22 11:45:11,2017-08-01 18:23:30,2017-07-26 18:09:10,2017-08-11
19268,5f9d46795c3126674e52becb3a1a517f,79287bcaafdde5c793b996fc40bb7d9f,delivered,2017-07-18 11:48:20,2017-07-18 12:03:29,2017-07-20 23:03:42,2017-07-20 18:52:41,2017-07-31


### Análise de duplicidades

In [63]:
df.duplicated().sum()

np.int64(0)